# Jitter Measurements
This Jupiter notebook measures the jitter of a SPAD system. The experiment uses PLD-800-D pulsed laser and the Swabian Time Tagger.  

## Countrate Calibration: SPAD vs LASER 
Live monitor of ```tt.Countrate(tagger, channels)``` to calibrate the counts per second (cps) of the SPAD array (SPAD A & B), and the laser pulse signal (SYNC). The system is calibrated when: 
* ```SYNC``` channel matches Laser Rep. Rate (see note below)
* ```SPAD$'s``` channels are attenuated with ND filters to a range between $ 1\% - 5\% $ of the SYNC channel. 

The LASER (PLD 800-D), is able to emmit pulses at the following frequencies: 
* Rep. Rate 1 = 1MHz    (One pulse each 1,000 ns)
* Rep. Rate 2 = 2MHz    (One pulse each 500 ns)
* Rep. Rate 4 = 4MHz    (One pulse each 250 ns)
* Rep. Rate 8 = 8MHz    (One pulse each 125 ns)
* Rep. Rate 16 = 16MHz  (One pulse each 62.5 ns)
* Rep. Rate 32 = 32 MHz (One pulse each 31.25 ns)





In [ ]:
import TimeTagger as tt
import numpy as np

# Experimental Parameters
SYNC_CH = 1             # Connected to PicoQuant 'SYNC OUT'
SPAD_A_CH = 2           # Connected to attenuated SPAD
SPAD_B_CH = 3
ADQ_TIME_PS = int(5e11) # 0.5 seconds integration for live feed.

# Initialize Time Tagger 
tagger = tt.createTimeTagger()
print(f"Connected to {tagger.getModel}")
print("--> SYNC Channel should perfectly match Laser Rep Rate.")
print("--> SPAD Channel MUST be attenuated to ~1% to 5% of the SYNC rate.")
# Measure countrates
cr = tt.Countrate(tagger, channels=[SYNC_CH, SPAD_A_CH, SPAD_B_CH]) 

# Initialize list
rates_sync_history = []
rates_spad_a_history = []
rates_spad_b_history = []
try: 
    while True: 
        cr.startFor(ADQ_TIME_PS, clear=True)    
        cr.waitUntilFinished()
        
        rates = cr.getData()      
        
        rate_sync = rates[0]        # Channel SYNC_CH
        rate_spad_a = rates[1]      # Channel SPAD_A_CH
        rate_spad_b = rates[2]      # Channel SPAD_B_CH
        
        rates_sync_history.append(rate_sync)
        rates_spad_a_history.append(rate_spad_a)
        rates_spad_b_history.append(rate_spad_b)
        
        print(f"\r[LIVE] SYNC: {rate_sync:.3f} cps  |  SPAD A: {rate_spad_a:.6f} cps   | SPAD B: {rate_spad_b:.6f} cps   ", end="", flush=True)

except KeyboardInterrupt:
    print("\n\nHalting Monitor...")
    
    np_rates_sync = np.array(rates_sync_history)
    np_rates_spad_a = np.array(rates_spad_a_history)
    np_rates_spad_b = np.array(rates_spad_b_history)

    
    if len(np_rates_sync) > 0:
        mean_sync = np.mean(np_rates_sync)
        mean_spad_a = np.mean(np_rates_spad_a)
        mean_spad_b = np.mean(np_rates_spad_a)
        ratio_a = (mean_spad_a / mean_sync) * 100
        ratio_b = (mean_spad_b / mean_sync) * 100
        
        print(f"--- FINAL STATISTICS ({len(np_rates_sync)} samples) ---")
        print(f"Mean SYNC Rate: {mean_sync:.3f} cps |\n Mean SPAD A Rate: {mean_spad_a:.6f} cps |\n SPAD B Rate: {mean_spad_b:.6f} cps")
        print(f"Photon Detection Ratios | \n SPAD A: {ratio_a:.2f}% | SPAD B: {ratio_b:.2f}% ")

finally: 
    tt.freeTimeTagger(tagger)
    print("Hardware Released.")

## Jitter Measurement
* Calibration ensures we operate in single photon regime. 
* Measurement is achieved with ```tt.Correlation(tagger, channel1, channel2)``` between the laser pulse signal (```SYNC```) and each SPAD. 
* The ideal SPAD (with zero jitter), will return a coincidence count signal $C(\tau)$ localized in a single bin whose height indicates the number of detected pulses, and whose time $\tau$ indicates the time of flight of the photon. 
* Because SPADs have inherent timing jitter, the measured delay $\tau$ fluctuates from one detection to another. This results in a normally distributed peak whose Full Width at Half Maximum (**FWHM**) mathematically quantifies the detector's jitter.
* Additionally, we measure the convolution of both detectors by comparing the correlation between the two SPAD's. The FWHM are related via: $FWHM_{AB} = \sqrt(FWHM_{A}^2+FWHM_{B}^2)$

In [ ]:
import TimeTagger as tt 
import numpy as np 
import matplotlib.pyplot as plt

#   EXPERIMENTAL PARAMETERS
binwidth_ps = 10    # pico-seconds
n_bins = 1000     
ADQ_TIME_PS = int(1e12)
SYNC_CH = 1             # Connected to PicoQuant 'SYNC OUT'
SPAD_A_CH = 2           # Connected to attenuated SPAD
SPAD_B_CH = 3

#   TIME TAGGER 
tagger = tt.createTimeTagger()
print(f"Connected to {tagger.getModel()}")
##  SETUP CORRELATION MEASUREMENTS
corr_SYNC_A = tt.Correlation(tagger,SYNC_CH, SPAD_A_CH,binwidth_ps, n_bins)
corr_SYNC_B = tt.Correlation(tagger,SYNC_CH, SPAD_B_CH,binwidth_ps, n_bins)
corr_A_B = tt.Correlation(tagger, SPAD_A_CH, SPAD_B_CH, binwidth_ps, n_bins)
##  RUN MEASUREMENTS IN PARELLEL 
corr_SYNC_A.startFor(ADQ_TIME_PS, clear = True)
corr_SYNC_B.startFor(ADQ_TIME_PS, clear = True)
corr_A_B.startFor(ADQ_TIME_PS, clear = True)
corr_SYNC_A.waitUntilFinished()
##  ADQUIRE MEASUREMENTS
corr_SYNC_A_data = corr_SYNC_A.getData()
corr_SYNC_B_data = corr_SYNC_B.getData()
corr_A_B_data = corr_A_B.getData()
time_array_ps = corr_SYNC_A.getIndex()  # Same array for all correlation experiments
tt.freeTimeTagger(tagger)


#   PLOT DATA
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))  # Create subplots

# SPAD A Plot
ax1.plot(time_array_ps , corr_SYNC_A_data, drawstyle='steps-mid', color='blue')
ax1.set_xlabel('Time Delay (ps)')
ax1.set_ylabel('Counts')
ax1.set_title(f'Autocorrelation: SPAD CH: {SPAD_A_CH} vs LASER {SYNC_CH} ')
#ax1.set_yscale('log')
#ax1.set_xlim(0, 10000) # Zoomed in to 30 ns
ax1.grid(True)

# SPAD B Plot
ax2.plot(time_array_ps , corr_SYNC_B_data, drawstyle='steps-mid', color='blue')
ax2.set_xlabel('Time Delay (ps)')
ax2.set_ylabel('Counts')
ax2.set_title(f'Autocorrelation: SPAD CH: {SPAD_B_CH} vs LASER {SYNC_CH} ')
#ax2.set_yscale('log')
#ax2.set_xlim(0, 10000) # Zoomed in to 30 ns
ax2.grid(True)

# SPADs Plot
ax3.plot(time_array_ps , corr_A_B_data, drawstyle='steps-mid', color='blue')
ax3.set_xlabel('Time Delay (ps)')
ax3.set_ylabel('Counts')
ax3.set_title(f'Autocorrelation: SPAD CH: {SPAD_B_CH} vs LASER {SYNC_CH} ')
#ax3.set_yscale('log')
#ax3.set_xlim(0, 10000) # Zoomed in to 30 ns
ax3.grid(True)

plt.tight_layout() # Automatically adjusts spacing so labels don't overlap
plt.show()


In [ ]:
#   Function to calculate FWHM
def calculate_fwhm(x_time, y_counts):
    max_idx = np.argmax(y_counts)   #  Find the maximum index
    y_max = y_counts[max_idx]       #  Save value
    
    #  Calculate baseline noise and half-max threshold
    baseline = np.mean(y_counts[:50]) 
    half_max = baseline + (y_max - baseline) / 2.0
    
    #  Find Left Crossing (Walk backward from the peak)
    left_idx = max_idx
    while left_idx > 0 and y_counts[left_idx] > half_max:
        left_idx -= 1
        
    # Linear interpolation for sub-bin precision on the left
    # Math: x = x0 + (y - y0) * (x1 - x0) / (y1 - y0)
    t_left = x_time[left_idx] + (half_max - y_counts[left_idx]) * (x_time[left_idx + 1] - x_time[left_idx]) / (y_counts[left_idx + 1] - y_counts[left_idx])
    
    #  Find Right Crossing (Walk forward from the peak)
    right_idx = max_idx
    while right_idx < len(y_counts) - 1 and y_counts[right_idx] > half_max:
        right_idx += 1
        
    # Linear interpolation for sub-bin precision on the right
    t_right = x_time[right_idx - 1] + (half_max - y_counts[right_idx - 1]) * (x_time[right_idx] - x_time[right_idx - 1]) / (y_counts[right_idx] - y_counts[right_idx - 1])
    
    # Calculate Final FWHM
    fwhm_ps = t_right - t_left
    
    return fwhm_ps, t_left, t_right, half_max

# Process SPAD A
fwhm_A, left_A, right_A, hmax_A = calculate_fwhm(time_array_ps, corr_SYNC_A_data)
print(f"SPAD A Pure Jitter:      {fwhm_A:.2f} ps")

# Process SPAD B
fwhm_B, left_B, right_B, hmax_B = calculate_fwhm(time_array_ps, corr_SYNC_B_data)
print(f"SPAD B Pure Jitter:      {fwhm_B:.2f} ps")

# Process Cross-Correlation (System Jitter)
fwhm_AB, left_AB, right_AB, hmax_AB = calculate_fwhm(time_array_ps, corr_A_B_data)
print(f"System (A+B) Jitter:     {fwhm_AB:.2f} ps")
print("--------------------------------")

# ==========================================
# MATHEMATICAL VERIFICATION
# ==========================================
# Since errors add in quadrature, FWHM_AB should theoretically be roughly equal to sqrt(FWHM_A^2 + FWHM_B^2)
theoretical_system_jitter = np.sqrt(fwhm_A**2 + fwhm_B**2)
error_margin = abs(fwhm_AB - theoretical_system_jitter) / theoretical_system_jitter * 100

print(f"Theoretical Conv. Jitter: {theoretical_system_jitter:.2f} ps")
print(f"Quadrature Error Margin:  {error_margin:.2f}%")